In [1]:
import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. 文件路径
# ============================================================

INPUT_FILE = Path(
    "/share/home/zhoujb/project/riceDataComb/processData/"
    "1-0-4-Soybean_2795/rawData/GSTP014.pheno"
)

OUTPUT_FILE = Path(
    "/share/home/zhoujb/project/riceDataComb/processData/"
    "1-0-4-Soybean_2795/phen_matrix_soybean.txt"
)

In [2]:
# ============================================================
# 2. 需要保留的性状
# ============================================================

TRAITS = [
    "BBD",
    "BPD",
    "MD",
    "FA16C",
    "FA18C",
    "FAC",
    "HSW",
    "SL",
    "ST",
    "LL",
    "LW",
    "NN",
    "OB",
    "PLH",
    "VBN",
    "PRT",
]


# ============================================================
# 3. 读取原始表型文件
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    na_values=["NA"]
)

print("Original shape:", df.shape)


# ============================================================
# 4. LINE 重命名为 ID
# ============================================================

df = df.rename(columns={"LINE": "ID"})


# ============================================================
# 5. 删除所有 BLUP 列
# ============================================================

blup_cols = [
    col for col in df.columns
    if col.endswith("_BLUP")
]

df = df.drop(columns=blup_cols)

print(f"Removed {len(blup_cols)} BLUP columns")


# ============================================================
# 6. 解析 trait / location / year
# ============================================================
#
# 普通列：
#
# BBD_BeiJing_2013
# → BBD / BeiJing / 2013
#
# 特殊列：
#
# BBD_ShanXi_2014_Drought
# → BBD / ShanXi_Drought / 2014
#
# BBD_ShanXi_2014_Wet
# → BBD / ShanXi_Wet / 2014
#
# ============================================================

column_info = []

for col in df.columns:

    if col == "ID":
        continue

    parts = col.split("_")

    trait = parts[0]

    # 只关注指定性状
    if trait not in TRAITS:
        continue

    # 找到年份所在位置
    year_idx = None

    for i, part in enumerate(parts):
        if re.fullmatch(r"\d{4}", part):
            year_idx = i
            break

    if year_idx is None:
        print(f"Warning: cannot parse year: {col}")
        continue

    year = parts[year_idx]

    # location位于 trait 和 year 之间
    location_parts = parts[1:year_idx]

    # 如果year后面还有Drought/Wet等环境信息，
    # 合并到location中
    if year_idx + 1 < len(parts):
        location_parts += parts[year_idx + 1:]

    location = "_".join(location_parts)

    column_info.append(
        {
            "column": col,
            "trait": trait,
            "location": location,
            "year": int(year),
        }
    )


column_info = pd.DataFrame(column_info)

print("\nParsed phenotype columns:")
print(column_info)

print(
    "\nNumber of parsed phenotype columns:",
    len(column_info)
)


# ============================================================
# 7. 宽表转换成长表
# ============================================================

phen_cols = column_info["column"].tolist()

long_df = df[
    ["ID"] + phen_cols
].melt(
    id_vars="ID",
    var_name="original_column",
    value_name="phenotype"
)


# ============================================================
# 8. 加入 trait / location / year 信息
# ============================================================

long_df = long_df.merge(
    column_info,
    left_on="original_column",
    right_on="column",
    how="left"
)

long_df = long_df[
    [
        "ID",
        "trait",
        "location",
        "year",
        "phenotype",
    ]
]


# ============================================================
# 9. 删除没有表型值的数据
# ============================================================

long_df = long_df.dropna(
    subset=["phenotype"]
)


# ============================================================
# 10. 转成 ID × location × year × trait 的矩阵
# ============================================================

phen_matrix = (
    long_df
    .pivot_table(
        index=[
            "ID",
            "location",
            "year",
        ],
        columns="trait",
        values="phenotype",
        aggfunc="first",
    )
    .reset_index()
)


# 去除 columns 的名字 "trait"
phen_matrix.columns.name = None


# ============================================================
# 11. 按指定性状顺序排列
# ============================================================

existing_traits = [
    trait
    for trait in TRAITS
    if trait in phen_matrix.columns
]

phen_matrix = phen_matrix[
    [
        "ID",
        "location",
        "year",
        *existing_traits,
    ]
]


# ============================================================
# 12. 删除所有性状都为空的环境行
# ============================================================

phen_matrix = phen_matrix.dropna(
    subset=existing_traits,
    how="all"
)


# ============================================================
# 13. 排序
# ============================================================

phen_matrix = phen_matrix.sort_values(
    by=[
        "ID",
        "location",
        "year",
    ]
).reset_index(drop=True)


# ============================================================
# 14. 输出
# ============================================================

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

phen_matrix.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False,
    na_rep="NA"
)


# ============================================================
# 15. 检查结果
# ============================================================

print("\nOutput shape:", phen_matrix.shape)

print("\nOutput columns:")
print(phen_matrix.columns.tolist())

print("\nLocations:")
print(sorted(phen_matrix["location"].unique()))

print("\nYears:")
print(sorted(phen_matrix["year"].unique()))

print("\nFirst 10 rows:")
print(phen_matrix.head(10))

print(f"\nSaved to:\n{OUTPUT_FILE}")

Original shape: (2795, 112)
Removed 15 BLUP columns

Parsed phenotype columns:
                     column trait        location  year
0          BBD_BeiJing_2013   BBD         BeiJing  2013
1          BBD_BeiJing_2014   BBD         BeiJing  2014
2            BBD_HeNan_2014   BBD           HeNan  2014
3            BBD_HeNan_2015   BBD           HeNan  2015
4     BBD_HeiLongJiang_2014   BBD    HeiLongJiang  2014
..                      ...   ...             ...   ...
89          VBN_ShanXi_2014   VBN          ShanXi  2014
90  VBN_ShanXi_2014_Drought   VBN  ShanXi_Drought  2014
91      VBN_ShanXi_2014_Wet   VBN      ShanXi_Wet  2014
92  VBN_ShanXi_2015_Drought   VBN  ShanXi_Drought  2015
93      VBN_ShanXi_2015_Wet   VBN      ShanXi_Wet  2015

[94 rows x 4 columns]

Number of parsed phenotype columns: 94

Output shape: (8600, 19)

Output columns:
['ID', 'location', 'year', 'BBD', 'BPD', 'MD', 'FA16C', 'FA18C', 'FAC', 'HSW', 'SL', 'ST', 'LL', 'LW', 'NN', 'OB', 'PLH', 'VBN', 'PRT']

Locati

In [3]:
phen_matrix

,ID,location,year,BBD,BPD,MD,FA16C,FA18C,FAC,HSW,SL,ST,LL,LW,NN,OB,PLH,VBN,PRT
0,s100,BeiJing,2013,61.0,NaN,155.0,27.1481,6.72891,208.714,15.1566,6.784,6.202,9.5000,6.26667,NaN,NaN,154.700,5.66667,340.553
1,s100,BeiJing,2014,55.0,NaN,168.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.26667,15.0,NaN,145.067,5.00000,NaN
2,s100,HeNan,2014,37.0,NaN,117.0,21.9953,6.58838,184.256,16.9720,7.126,6.544,10.2200,7.19333,NaN,NaN,130.000,1.80000,358.104
3,s100,HeNan,2015,43.0,57.0,115.0,NaN,NaN,NaN,18.1685,7.316,6.594,10.2133,4.00000,NaN,NaN,116.000,3.40000,NaN
4,s100,ShanXi,2013,59.0,61.0,NaN,NaN,NaN,NaN,17.0600,NaN,NaN,NaN,NaN,NaN,NaN,131.100,3.00000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8595,s99,ShanXi,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.300,2.00000,NaN
8596,s99,ShanXi_Drought,2014,50.0,NaN,126.0,NaN,NaN,NaN,20.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8597,s99,ShanXi_Drought,2015,33.0,NaN,88.0,NaN,NaN,NaN,13.3000,NaN,NaN,NaN,NaN,NaN,NaN,30.400,0.80000,NaN
8598,s99,ShanXi_Wet,2014,50.0,NaN,115.0,NaN,NaN,NaN,20.7000,NaN,NaN,NaN,NaN,NaN,620.0,57.800,2.80000,NaN


In [13]:
phen_matrix[(phen_matrix["ID"]=="s63")&(phen_matrix["location"]=="BeiJing")&(phen_matrix["year"]==2013)]

,ID,location,year,BBD,BPD,MD,FA16C,FA18C,FAC,HSW,SL,ST,LL,LW,NN,OB,PLH,VBN,PRT
6359,s63,BeiJing,2013,35.0,NaN,108.0,30.9182,7.31161,227.418,11.9398,7.46,6.244,7.96667,5.9,NaN,NaN,24.0667,2.0,397.82
